# Context Assembly for Agent Memory

Companion notebook for the Oracle AI Database article *Context Assembly for Multi-Tenant Agent Memory: Budgets, Selection, and Tool Calls with MCP*. It builds on the schema from From Prompt to Persistence [part 1](https://blogs.oracle.com/developers/from-prompt-to-persistence-part-1-designing-multi-tenant-agent-memory-schemas-for-saas) and [part 2](https://blogs.oracle.com/developers/from-prompt-to-persistence-part-2-putting-the-multi-tenant-agent-memory-schema-to-work), the canonical/derived split from [Persistent Memory and Derived Context](https://blogs.oracle.com/developers/persistent-memory-and-derived-context-a-two-layer-pattern-for-agents), and the retrieval pipeline from [Hybrid Retrieval for Agent Memory](https://blogs.oracle.com/developers/hybrid-retrieval-for-agent-memory-vector-lexical-and-metadata-together). Their companion notebooks are [multitenant_schema_walkthrough.ipynb](../multitenant_schema_walkthrough.ipynb), [two_layer_pattern_walkthrough.ipynb](../two_layer_pattern_walkthrough.ipynb), and [hybrid_retrieval_pipeline.ipynb](../hybrid_retrieval_pipeline.ipynb).

Hybrid retrieval produces ranked candidates. The next step is deciding which memories fit in the model request, how much space each memory type gets, and which details the agent can fetch later through tools. A historical summary can still contain an old claim, so the assembled context also needs its current canonical correction.

The notebook runs that assembly against a live database. It builds a window for Jane's Letta/AgentCore comparison and another for her Helios-RAG fact check, using different budget allocations. It exercises the Memory Manager through FastMCP, then replays a recorded request using its manifest and the retained source rows. The full earlier schema and seed payloads carry over; setup creates missing objects and adds rows without dropping tables or changing existing definitions. All examples are synthetic.

## Prerequisites

- Oracle AI Database 26ai, with `DB_USER`, `DB_PASSWORD`, and `DB_DSN` in an ignored repository-root `.env`.
- A schema owner with privileges to create tables, indexes, procedures, and trusted application contexts, plus access to `DBMS_RLS` and `DBMS_FLASHBACK`. The setup below uses [prerequisites.sql](prerequisites.sql) to grant the required privileges to the existing user configured as `DB_USER`; that user also needs a tablespace quota. The owner must not have `EXEMPT ACCESS POLICY`, because the examples exercise tenant row-level security.
- Outbound HTTPS from the notebook to Oracle Object Storage. Section 1 downloads the augmented `ALL_MINILM_L12_V2` embedding model if it isn't already loaded.
- A reranking model loaded as `BGE_RERANKER`, or an OML4Py export of `BAAI/bge-reranker-base` supplied through `RERANKER_ONNX_FILE`. The [reranker export notebook](../vecdb/reranking/load-model-for-in-database-reranking.ipynb) documents the conversion.
- From the repository root, open the notebook with `jupyter lab notebooks/context_assembly/context_assembly_walkthrough.ipynb`. The first code cell installs the pinned Python packages into the running kernel, including FastMCP and python-oracledb.
- For the optional model experiments in section 8, supply `OPENAI_API_KEY` and `OPENAI_MODEL` in an ignored notebook-local `.env`, and set `CONTEXT_RUN_MODEL=1`. Otherwise, that section displays recorded provider results.

Vectors are `VECTOR(384, FLOAT32)` to match `ALL_MINILM_L12_V2`. The [README](README.md) has the command-line runner, source provenance, and deployment limitations.


## Environment & connection

### Choose a database

Use **Oracle AI Database 26ai Free in Docker** for a local installation, or an existing **Oracle AI Database 26ai** instance whose administrator can provide the required privileges and resources. The local instructions below use the ARM64 image for database version `23.26.3.0.0`. Other architectures need a matching image from [Oracle's container registry](https://container-registry.oracle.com/ords/ocr/ba/database/free) and their own validation run.

**FreeSQL cannot run this walkthrough:** loading the required ONNX models needs `CREATE MINING MODEL`, and the tenant isolation setup needs `CREATE ANY CONTEXT`. FreeSQL does not provide these permissions.

### 1. Start a local database

Install and start Docker Desktop. In a **Bash** terminal, pull the image and start a dedicated tutorial database. Enter a new administrator password at the prompt; the value is passed through the environment rather than included in the command history.

```bash
IMAGE=container-registry.oracle.com/database/free:23.26.3.0-lite-arm64
docker pull "$IMAGE"
read -r -s -p "Database administrator password: " ORACLE_PWD
printf '\n'
export ORACLE_PWD
docker run -d --name context-assembly-free \
  -p 127.0.0.1:1521:1521 \
  -e ORACLE_PWD \
  -v context-assembly-data:/opt/oracle/oradata \
  "$IMAGE"
unset ORACLE_PWD
docker logs -f context-assembly-free
```

Wait for `DATABASE IS READY TO USE!`, then press Ctrl-C to leave the log stream. For later sessions, start the same database with `docker start context-assembly-free`; the named volume retains its data. If port 1521 is occupied, change the host port in both the command and `DB_DSN` below.

### 2. Grant privileges to your existing database user

Use the same database user configured as `DB_USER` in the repository-root `.env`, as in the other memory notebooks. The notebook assumes this account already exists; it does not require a particular username. For a new container, have an administrator provision your application account before continuing.

The user must be non-SYS and must not have `EXEMPT ACCESS POLICY`, because the notebook checks tenant row-level security. Setup creates the database-wide `memory_ctx` application context bound to this user's package. Reuse the owner of that context if it already exists.

Ask an administrator to apply [prerequisites.sql](prerequisites.sql) to this user in the same pluggable database used by `DB_DSN`. For the local container, run these commands from `notebooks/context_assembly`:

```bash
docker cp prerequisites.sql context-assembly-free:/tmp/context-prerequisites.sql
docker exec -it context-assembly-free \
  sqlplus / as sysdba
```

At the SQL*Plus prompt, select the PDB and run the grants script:

```sql
ALTER SESSION SET CONTAINER=FREEPDB1;
@/tmp/context-prerequisites.sql
```

Enter your existing `DB_USER` when prompted. The script grants Oracle Text access, model-loading privileges, trusted-context privileges, and access to `DBMS_RLS`, `DBMS_FLASHBACK`, and `DBMS_VECTOR`. It does not create another user or tablespace. The administrator must also ensure sufficient quota in the user's existing default tablespace; the local validation used a 1 GB quota.

The HNSW indexes need vector memory. The local lite image starts with `vector_memory_size=0`; the validation run used **128 MB**. For a new, dedicated local container, configure this once by connecting as SYS to the root container:

```bash
docker exec -it context-assembly-free sqlplus / as sysdba
```

```sql
SHOW PARAMETER vector_memory_size
-- Run the following only if vector memory has not already been configured.
ALTER SYSTEM SET vector_memory_size=128M SCOPE=SPFILE;
SHUTDOWN IMMEDIATE;
STARTUP;
EXIT
```

This restarts the local database. For an existing instance, have its administrator check and configure vector memory for the workload.

### 3. Configure the connection

The notebook's first `%pip` cell installs the required Python packages into the running kernel.

Reuse `DB_USER`, `DB_PASSWORD`, and `DB_DSN` from the ignored **repository-root** `.env`. If these values are not configured yet, add your existing user's credentials:

```dotenv
DB_USER=<your existing database username>
DB_PASSWORD=<your existing database password>
DB_DSN=localhost:1521/FREEPDB1
```

`FREEPDB1` is the pluggable database containing the tutorial objects; `FREE` connects to the root container. For an existing database, use the credentials and DSN supplied by its administrator. The DSN can be an Easy Connect string such as `hostname:1521/service_name` or a full connect descriptor. Neither credentials nor connection values are printed by the notebook.

### 4. Prepare the reranker model

Section 1 downloads and loads the augmented `ALL_MINILM_L12_V2` embedding model when absent. The Python host needs outbound HTTPS to Oracle Object Storage; Python uploads the model as a BLOB, so this download does not require a database network ACL.

For `BGE_RERANKER`, use the [reranker export notebook](../vecdb/reranking/load-model-for-in-database-reranking.ipynb) to prepare its Linux x86_64 OML4Py conversion environment and export `BAAI/bge-reranker-base`. Its conversion and local validation steps produce the augmented ONNX file needed here; this walkthrough does not require its VecDB REST, OCI upload, or PAR steps. The export configuration is:

```python
from oml.utils import MiningFunction, ONNXPipeline, ONNXPipelineConfig

model_id = "BAAI/bge-reranker-base"
config = ONNXPipelineConfig.from_preconfigured(
    model_id, max_seq_length=256, quantize_model=True, use_float16=False,
)
pipeline = ONNXPipeline(model_id, config=config, function=MiningFunction.REGRESSION)
pipeline.export2file("bge_reranker_base", "/work")
```

Copy the resulting file to the Python host and add its absolute path to an ignored **notebook-local** `.env`:

```dotenv
RERANKER_ONNX_FILE=/absolute/path/to/bge_reranker_base.onnx
```

The loader reuses compatible models already present in the owner schema. A Hugging Face weights file alone is not the required augmented ONNX graph.

### 5. Run the notebook

Open this notebook in your Jupyter environment. Run the `%pip` cell first to install the pinned packages into the selected kernel. If it replaces packages already imported in the session, restart the kernel and run all cells from the top. The following cell sets the Developer Hub program identifier before any database connection is created.

Section 1 verifies a 384-dimensional embedding and a reranker prediction. Section 7 checks tenant isolation, database behavior, and FastMCP calls; section 9 should report `hash_matches: true` for replay while the required undo remains available. Optional model-provider calls stay disabled unless `CONTEXT_RUN_MODEL=1` is set.



In [1]:
%pip install --quiet --disable-pip-version-check -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json
from pathlib import Path
import oracledb
oracledb.defaults.program = "devrel-developerhub-context-assembly-for-agent-memory"
from config import SCOPE, AS_OF, QUESTION, QUESTION_1, dumps
from store import OracleStore
from manager import MemoryManager
from model_setup import ensure_models
print(json.dumps({"scope": SCOPE, "evaluation_time_utc": AS_OF.isoformat()}, indent=2))

{
  "scope": {
    "tenant_id": "acme",
    "user_id": "user:jane@acme.com",
    "agent_id": "agent:research_v1",
    "thread_id": "thread_demo"
  },
  "evaluation_time_utc": "2026-09-28T12:00:00"
}


## 1. Load and verify the models

Every embedding is generated inside Oracle with `VECTOR_EMBEDDING`, using the augmented `ALL_MINILM_L12_V2` ONNX model. Tokenization is part of the model graph, so the function takes plain text and returns a 384-dimensional vector. `BGE_RERANKER` scores each query/chunk pair inside the database.

The loader reuses both models if they're already present. It downloads the embedding model when missing and loads the reranker from `RERANKER_ONNX_FILE` when needed. The cell checks the embedding dimensions and runs a sample prediction through the reranker.


In [3]:
print(json.dumps(ensure_models(), indent=2))

[
  {
    "model": "ALL_MINILM_L12_V2",
    "function": "EMBEDDING",
    "created": "2026-08-31T18:09:46"
  },
  {
    "model": "BGE_RERANKER",
    "function": "REGRESSION",
    "created": "2026-09-25T00:03:52"
  }
]


## 2. The schema and tenant boundary

`schema.sql` carries over all ten original tables and their indexes, plus the Oracle Text chunk index introduced by Hybrid Retrieval. Setup creates missing objects and applies tenant row-level security to the explicit table list. It stops if an existing table has different columns.

The new tables, `context_compression` and `context_source_ref`, store generated compressions and their source dependencies. The Memory Manager reads each memory type's existing fields and normalizes them in Python for assembly.

Persona rows, conversation events, and knowledge-base chunks have no version column, so their manifest references retain null versions. Chunks inherit scope and lifecycle from their parent document.


In [4]:
store = OracleStore()
print(json.dumps(store.setup(), indent=2))
with store.connection() as conn, conn.cursor() as cur:
    cur.execute("SELECT table_name, COUNT(*) FROM user_tab_columns WHERE table_name IN ('GUIDELINE_MEMORY','PERSONA_MEMORY','ENTITY_MEMORY','CONVERSATION_MEMORY','SUMMARIZATION_MEMORY','WORKFLOW_MEMORY','TOOLBOX_MEMORY','KNOWLEDGE_BASE_DOCUMENT','KNOWLEDGE_BASE_CHUNK','DELETION_EVENTS','CONTEXT_COMPRESSION','CONTEXT_SOURCE_REF') GROUP BY table_name ORDER BY table_name")
    print(json.dumps(dict(cur.fetchall()), indent=2))

{
  "base_tables": 10,
  "additional_tables": [
    "CONTEXT_COMPRESSION",
    "CONTEXT_SOURCE_REF"
  ],
  "existing_tables_altered": false
}
{
  "CONTEXT_COMPRESSION": 16,
  "CONTEXT_SOURCE_REF": 11,
  "CONVERSATION_MEMORY": 14,
  "DELETION_EVENTS": 7,
  "ENTITY_MEMORY": 20,
  "GUIDELINE_MEMORY": 14,
  "KNOWLEDGE_BASE_CHUNK": 9,
  "KNOWLEDGE_BASE_DOCUMENT": 17,
  "PERSONA_MEMORY": 14,
  "SUMMARIZATION_MEMORY": 22,
  "TOOLBOX_MEMORY": 18,
  "WORKFLOW_MEMORY": 21
}


## 3. The original examples

The baseline seed retains the Stripe examples and infrastructure supersession from From Prompt to Persistence, along with the Helios-RAG correction from 0.92 to 0.78. The fact keeps its original `model:helios-rag` subject and `beir_ndcg10` predicate. Hybrid Retrieval's research corpus carries over in full: 23 documents and 137 chunks under the original `tenant_acme` identifier.

Row IDs are stable and UTC seed dates are fixed so reruns are reproducible. The original text, schema fields, and identity namespaces are retained. `baseline_seed.py`, `research_corpus.py`, and `source_provenance.json` record the source cells and adaptations. The earlier notebooks' erasure and teardown steps are excluded.


In [5]:
print(json.dumps(store.seed_original(), indent=2))

{
  "schema_seed": "original Stripe payloads",
  "two_layer_seed": "original Helios payloads",
  "hybrid_seed": {
    "tenant": "tenant_acme",
    "documents": 23,
    "chunks": 137
  }
}


## 4. Additional seed data

Context assembly needs a budget policy and executable tools. This step adds the policy and the implemented toolbox entries, along with a July summary that still contains the old Helios-RAG score. Provenance links connect that history to its source; separate isolation cases exercise the tenant boundary.

An identical copy of the research corpus is added under `acme` so Jane can use it alongside the earlier memory examples. The original `tenant_acme` corpus remains present. Jane keeps `user:jane@acme.com`, and the research assistant uses `agent:research_v1`. The original facts, corrected summary, tool definition, and document text remain available.


In [6]:
print(json.dumps(store.seed_additional(), indent=2))

{
  "additional": "context budget, four tools, July summary, provenance, isolation rows",
  "corpus_copy": {
    "from": "tenant_acme",
    "to": "acme",
    "documents": 23,
    "chunks": 137
  },
  "original_rows_replaced": false
}


## 5. Hybrid retrieval

The query combines vector and Oracle Text candidate pools with reciprocal rank fusion, then applies the in-database BGE reranker. All source rows are read at one captured system change number (SCN). Parent-document predicates enforce chunk scope and lifecycle. Collection routing selects the Helios benchmark document for the fact check and the research corpus for the comparison.

BGE returns measured logits. Sigmoid normalization turns them into bounded weights for assembly; those weights aren't probabilities of correctness. The query scans the small tutorial corpus directly, so its execution time doesn't establish production performance.


In [7]:
hits = store.hybrid(question=QUESTION_1)
print(json.dumps([{k: h[k] for k in ("title", "rrf_score", "rerank_raw")} for h in hits[:5]], indent=2))
print(Path("hybrid_query.sql").read_text())

[
  {
    "title": "Letta: Agent Memory Beyond the Context Window",
    "rrf_score": 0.03201844262295082,
    "rerank_raw": -0.48926571011543274
  },
  {
    "title": "Letta: Agent Memory Beyond the Context Window",
    "rrf_score": 0.03125,
    "rerank_raw": -0.5860196352005005
  },
  {
    "title": "Letta: Agent Memory Beyond the Context Window",
    "rrf_score": 0.015625,
    "rerank_raw": -2.992020606994629
  },
  {
    "title": "Letta: Agent Memory Beyond the Context Window",
    "rrf_score": 0.03055037313432836,
    "rerank_raw": -3.1146624088287354
  },
  {
    "title": "Amazon Bedrock AgentCore Memory Developer Guide",
    "rrf_score": 0.028373015873015873,
    "rerank_raw": -3.4837546348571777
  }
]
WITH vec_pool AS (
  SELECT c.id, ROW_NUMBER() OVER (
           ORDER BY VECTOR_DISTANCE(c.embedding,
             VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE), c.id) AS vec_rank
    FROM knowledge_base_chunk AS OF SCN :scn c
    JOIN knowledge_base_document A

## 6. Assemble both windows

The Memory Manager builds a comparison window and a fact-check window from the same scoped store. The comparison allocates more space to retrieved evidence. The fact check includes the current canonical correction at full size whenever it annotates July history. Assembly fails if that required fact cannot fit.

Each memory record has an ID and a type, plus a status that distinguishes current facts from historical claims. This OpenAI example follows the [prompt engineering guide](https://developers.openai.com/api/docs/guides/prompt-engineering#message-formatting-with-markdown-and-xml): Markdown headings organize developer instructions, and XML tags delimit memory records in the user context. The question follows the context. Tool definitions and later tool results retain their native API structures.

`status` describes the record's role in this window: `current` for active canonical facts and workflows, `historical` for episode summaries, `unverified` for recalled conversation, and `evidence` for retrieved documents. It doesn't prove a claim is correct. Assembler annotations are separate from the original content.

The manifest maps each citation ID to its stored memory and source lineage. It records ranking scores and dependencies, along with budget allocations, exclusions, rendering tiers, and code versions. The question is referenced by its source event instead of stored again. Citation presence and citation support need separate checks; rendering IDs doesn't guarantee correct citations.


In [8]:
manager = MemoryManager(store)
comparison = manager.build_window(question=QUESTION_1, intent="comparison")
fact_check = manager.build_window()
for name, window in (("comparison", comparison), ("fact_check", fact_check)):
    print(name)
    print(json.dumps({k: window["manifest"][k] for k in ("scn", "request_hash", "local_text_tokens", "required_refs", "allocation")}, indent=2))
print(fact_check["request"]["input"][0]["content"])
print("<memory_context>")
for b in fact_check["memories"]:
    if b["ref"][1] == "context_july_summary" or b["ref"] in fact_check["manifest"]["required_refs"]:
        print(b["text"])
print("</memory_context>")
print("# Current question\n" + QUESTION)

comparison
{
  "scn": 4366749,
  "request_hash": "7181ba3682f05d14d17226bcccb1bc97df017e5a49972917186211728f6b5e5f",
  "local_text_tokens": 2873,
  "required_refs": [
    [
      "entity",
      "ctxseed_2688f0418960d68379370cf8",
      2
    ]
  ],
  "allocation": {
    "evidence": {
      "base": 1479,
      "carry_in": 0,
      "used": 1451
    },
    "durable_facts": {
      "base": 369,
      "carry_in": 28,
      "used": 284
    },
    "volatile_tail": {
      "base": 493,
      "carry_in": 113,
      "used": 253
    },
    "recalled_history": {
      "base": 123,
      "carry_in": 353,
      "used": 234
    }
  }
}
fact_check
{
  "scn": 4366767,
  "request_hash": "d6ff84a7f5d2f2e60a9829422bdbcc8ae97114f5033d7603ef92d9c54c5a0f4d",
  "local_text_tokens": 1636,
  "required_refs": [
    [
      "entity",
      "ctxseed_2688f0418960d68379370cf8",
      2
    ]
  ],
  "allocation": {
    "evidence": {
      "base": 615,
      "carry_in": 0,
      "used": 209
    },
    "durable_facts"

## 7. Database and tool checks

The following cell checks tenant reads and writes, including connection-context cleanup. It also exercises embedding refresh and pointer expansion, rechecks active tools, and verifies that fixed-slice and tool-loop budgets reject requests that exceed their limits.

The replay check temporarily deletes an annotation source. Replay must refuse that dependency even while Flashback can recover its old bytes. A `finally` block restores the row. A separate controlled ablation gives durable facts zero space and disables admission of the current correction; the default assembler rejects that configuration. Optional evidence can use the pointer tier under a separate narrow budget.

FastMCP runs the clarification checks in both legacy handshake mode and protocol `2026-07-28`. Each mode covers accept, decline, cancel, and absent elicitation support. Calls that try to override the tenant or guess a handle must return `invalid_arguments` or `access_denied`. Those MCP errors are expected test results; unexpected exceptions still produce server tracebacks.


In [9]:
from run_checks import database_checks, protocol_checks
verification, checked_window, checked_comparison = database_checks(store)
verification["fastmcp"] = await protocol_checks(checked_window)
Path("results").mkdir(exist_ok=True)
Path("results/verification.json").write_text(json.dumps(json.loads(dumps(verification)), indent=2) + "\n")
for name, value in (("fact_check_window", checked_window), ("comparison_window", checked_comparison)):
    Path("results", name + ".json").write_text(json.dumps(json.loads(dumps(value)), indent=2) + "\n")
print(json.dumps(verification, indent=2))

{
  "database_version": "23.26.3.0.0",
  "isolation": {
    "acme": 9,
    "globex": 1,
    "tenant_acme": 0,
    "no_context": 0
  },
  "request_hash": "d6ff84a7f5d2f2e60a9829422bdbcc8ae97114f5033d7603ef92d9c54c5a0f4d",
  "local_text_tokens": 1636,
  "corrected_fact_required": true,
  "full_research_corpus_preserved": true,
  "guideline_version_update": true,
  "fixed_and_loop_budget_failures": true,
  "historical_dependency_deletion_blocks_replay": true,
  "flashback_recovers_pre_deletion_bytes": true,
  "embedding_distance_current_vs_old": [
    0.0,
    0.016072721954478175
  ],
  "active_tool_rechecked": true,
  "tool_authorization_uses_present_time": true,
  "derived_write_nondefault_tenant": true,
  "valid_pointer_expands": true,
  "required_fact_budget_failure": true,
  "controlled_ablation_omits_current": true,
  "comparison_includes_both_sources": true,
  "replay": {
    "hash_matches": true,
    "request_hash": "d6ff84a7f5d2f2e60a9829422bdbcc8ae97114f5033d7603ef92d9c54c5a0f4

## 8. Inspect or rerun the model experiments

By default, this cell displays recorded provider results. Compression replay is revalidated only when the referenced compression and source rows are available in the connected database; otherwise, the output labels revalidation as skipped and retains the archived results. Set `CONTEXT_RUN_MODEL=1` to generate a stored compression and rerun the model calls through FastMCP. Each compression retains its source identity and hash. It also records the generating model, prompt version, and provider response so replay can reuse the stored text.

The missing-detail experiment uses a narrow evidence budget that omits AgentCore's retention paragraph. The model must fetch that detail through the hybrid query. The current canonical correction remains required.

The recorded fact-check run returned 0.78 but omitted memory citations. The fetched-detail run supplied memory IDs; the latest comparison run also omitted them. These results came from the earlier renderer. Rerunning the cell with `CONTEXT_RUN_MODEL=1` produces results for the current renderer. Citation presence is checked separately from whether a citation supports a claim.

Local token estimates include a framing reserve. Set `CONTEXT_PROVIDER_COUNT=1` to use the optional provider counter.


In [ ]:
if os.getenv("CONTEXT_RUN_MODEL") == "1":
    from model_checks import main as run_model_checks
    await run_model_checks()
record = Path("results/model_verification.json")
if record.exists():
    results = json.loads(record.read_text())
    compression = results.get("compression", {})
    dependencies = [compression.get("compression_ref"), compression.get("source_ref")]
    if all(dependencies) and all(store.source_available(reference) for reference in dependencies):
        from model_checks import revalidate_stored_compression
        results = revalidate_stored_compression(store, results)
        revalidation = {"status": "passed", "database": "connected"}
    else:
        revalidation = {
            "status": "skipped",
            "reason": "Referenced compression or source rows are unavailable in this database; results below are archived.",
        }
    print(json.dumps({"mode": "recorded_provider_results", "compression_revalidation": revalidation, "results": results}, indent=2))
else:
    print("Model experiments were not run; set CONTEXT_RUN_MODEL=1 and supply model settings.")


## 9. Replay and close

Replay first checks current authorization and deletion, including transitive provenance. It reads retained source rows at the captured SCN, uses recorded ranking scores and stored compression, and verifies the request hash. Missing undo or a deleted dependency makes replay unavailable. Tool-round replay also requires the retained provider transcript.

Manifests, stored compressions, provenance links, and recorded responses need the same retention and deletion controls as their sources in a deployment.

In [11]:
from replay import replay_base_window
print(json.dumps(replay_base_window(store, checked_window), indent=2))
store.close()

{
  "hash_matches": true,
  "request_hash": "d6ff84a7f5d2f2e60a9829422bdbcc8ae97114f5033d7603ef92d9c54c5a0f4d",
  "scn": 4366784,
  "scores": "recorded",
  "authorization": "current"
}
